In [1]:
import sys
sys.path.append("..")

import json
import joblib
import lightgbm
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.pipeline import Pipeline

from src.db import get_engine
from src.features import (
    build_features, build_features_v3, get_numeric_features,
    CATEGORICAL_FEATURES_V3, BINARY_FEATURES,
)
from src.models import build_lightgbm_preprocessor, build_lightgbm_model, CalibratedModel, logit
from src.policy import decide

titles = pd.read_sql(
    "SELECT TRY_CAST(id AS bigint) AS id, emp_title FROM dbo.loans_population",
    get_engine(),
)

data = {}
for name in ["train", "valid", "test"]:
    data[name] = pd.read_parquet(f"../data/processed/{name}.parquet").merge(titles, on="id", how="left")

data["train"] = data["train"][data["train"]["issue_date"] >= "2013-01-01"].reset_index(drop=True)

X = {name: df.drop(columns=["target"]) for name, df in data.items()}
y = {name: df["target"] for name, df in data.items()}
print({name: len(df) for name, df in data.items()})

{'train': 262992, 'valid': 120790, 'test': 162236}


In [2]:
numeric_features = get_numeric_features(
    build_features(X["train"].drop(columns=["emp_title"]).head(1000))
)

prep = build_lightgbm_preprocessor(
    numeric_features, CATEGORICAL_FEATURES_V3, BINARY_FEATURES,
    feature_function=build_features_v3,
)
X_tr = prep.fit_transform(X["train"])
X_va = prep.transform(X["valid"])

model = build_lightgbm_model(num_leaves=15, min_child_samples=500,
                             cat_smooth=100, min_data_per_group=1000)
model.fit(
    X_tr, y["train"],
    eval_X=X_va,
    eval_y=y["valid"],
    eval_metric="auc",
    categorical_feature=CATEGORICAL_FEATURES_V3,
    callbacks=[lightgbm.early_stopping(stopping_rounds=100, verbose=False)],
)
pipeline_v3 = Pipeline([("prep", prep), ("model", model)])

raw_valid = pipeline_v3.predict_proba(X["valid"])[:, 1]
calibrator = LogisticRegression().fit(logit(raw_valid).reshape(-1, 1), y["valid"])
model_v3 = CalibratedModel(pipeline_v3, calibrator)

p_valid = model_v3.predict_proba(X["valid"])[:, 1]
p_test = model_v3.predict_proba(X["test"])[:, 1]
print("Trees:", model.best_iteration_)
print("Validation AUC:", round(roc_auc_score(y["valid"], p_valid), 4))
print("Test AUC:", round(roc_auc_score(y["test"], p_test), 4))

Trees: 694
Validation AUC: 0.6966
Test AUC: 0.7046


In [3]:
v2 = joblib.load("../models/lightgbm_v2_calibrated.joblib")
logistic = joblib.load("../models/logistic_baseline.joblib")

X_test_old = X["test"].drop(columns=["emp_title"])
p_test_v2 = v2.predict_proba(X_test_old)[:, 1]
p_test_logistic = logistic.predict_proba(X_test_old)[:, 1]

sub_grade_order = {f"{g}{n}": (i - 1) * 5 + n
                   for i, g in enumerate("ABCDEFG", start=1) for n in range(1, 6)}
score_sub_grade = data["test"]["sub_grade"].map(sub_grade_order).values


def ks_statistic(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return (tpr - fpr).max()


scores = {
    "LightGBM v3": p_test,
    "LightGBM v2": p_test_v2,
    "Logistic regression": p_test_logistic,
    "Lending Club sub_grade": score_sub_grade,
}
comparison = pd.DataFrame({
    name: {"auc": roc_auc_score(y["test"], s), "ks": ks_statistic(y["test"], s)}
    for name, s in scores.items()
}).T
comparison["gini"] = 2 * comparison["auc"] - 1
print(comparison.round(4))

rng = np.random.default_rng(42)
y_arr = y["test"].values
lead_grade, lead_v2 = [], []
for _ in range(200):
    s = rng.integers(0, len(y_arr), size=len(y_arr))
    auc_v3 = roc_auc_score(y_arr[s], p_test[s])
    lead_grade.append(auc_v3 - roc_auc_score(y_arr[s], score_sub_grade[s]))
    lead_v2.append(auc_v3 - roc_auc_score(y_arr[s], p_test_v2[s]))
lead_grade, lead_v2 = np.array(lead_grade), np.array(lead_v2)

for label, lead in [("Lending Club sub_grade", lead_grade), ("v2", lead_v2)]:
    low, high = np.percentile(lead, [2.5, 97.5])
    print(f"v3 lead over {label}: {lead.mean():.4f} (95% {low:.4f} to {high:.4f}), "
          f"ahead in {(lead > 0).mean() * 100:.0f}%")

                           auc      ks    gini
LightGBM v3             0.7046  0.2963  0.4092
LightGBM v2             0.7026  0.2941  0.4051
Logistic regression     0.6871  0.2706  0.3741
Lending Club sub_grade  0.6814  0.2665  0.3629
v3 lead over Lending Club sub_grade: 0.0231 (95% 0.0205 to 0.0262), ahead in 100%
v3 lead over v2: 0.0020 (95% 0.0010 to 0.0031), ahead in 100%


In [4]:
profit = {n: (data[n]["total_pymnt"] - data[n]["funded_amnt"]).values for n in ["valid", "test"]}
thresholds = np.round(np.arange(0.05, 0.605, 0.005), 3)


def best_cutoff(p_valid_scores):
    totals = [profit["valid"][p_valid_scores < t].sum() for t in thresholds]
    return float(thresholds[int(np.argmax(totals))])


def apply_policy(p, cutoff):
    approved = p < cutoff
    return {
        "approval_rate_pct": approved.mean() * 100,
        "default_rate_approved_pct": y["test"].values[approved].mean() * 100,
        "profit_millions": profit["test"][approved].sum() / 1e6,
    }


cutoff_v3 = best_cutoff(p_valid)
cutoff_logistic = best_cutoff(logistic.predict_proba(X["valid"].drop(columns=["emp_title"]))[:, 1])
cutoff_v2 = json.load(open("../results/policy.json"))["cutoff"]

policy_table = pd.DataFrame({
    "Approve everyone": apply_policy(p_test, 1.01),
    "Logistic regression policy": apply_policy(p_test_logistic, cutoff_logistic),
    "LightGBM v2 policy": apply_policy(p_test_v2, cutoff_v2),
    "LightGBM v3 policy": apply_policy(p_test, cutoff_v3),
}).T
policy_table["gain_vs_approve_all"] = (
    policy_table["profit_millions"] - policy_table.loc["Approve everyone", "profit_millions"]
)
print("v3 cut-off (chosen on validation):", cutoff_v3)
print(policy_table.round(2))

REVIEW_MARGIN = 0.03
decisions = pd.Series([decide(p, cutoff_v3, REVIEW_MARGIN) for p in p_test])
decision_summary = pd.DataFrame({
    "loans": decisions.value_counts(),
    "default_rate_pct": pd.Series(y_arr).groupby(decisions).mean() * 100,
    "profit_millions": pd.Series(profit["test"]).groupby(decisions).sum() / 1e6,
}).loc[["Approve", "Review", "Reject"]]
decision_summary.round(2)

v3 cut-off (chosen on validation): 0.34
                            approval_rate_pct  default_rate_approved_pct  \
Approve everyone                       100.00                      14.71   
Logistic regression policy              98.45                      14.32   
LightGBM v2 policy                      97.18                      13.95   
LightGBM v3 policy                      95.95                      13.68   

                            profit_millions  gain_vs_approve_all  
Approve everyone                     132.20                 0.00  
Logistic regression policy           132.81                 0.61  
LightGBM v2 policy                   134.75                 2.55  
LightGBM v3 policy                   135.46                 3.26  


,loans,default_rate_pct,profit_millions
Approve,152090,13.24,135.45
Review,3579,32.52,0.00
Reject,6567,39.06,-3.26


In [5]:
joblib.dump(model_v3, "../models/lightgbm_v3_calibrated.joblib")

policy_v3 = {
    "model": "lightgbm_v3_calibrated",
    "cutoff": cutoff_v3,
    "review_margin": REVIEW_MARGIN,
    "chosen_on": "validation (2015-01 to 2015-06)",
    "trained_on": "2013-01 to 2014-12, with job family",
}
with open("../results/policy_v3.json", "w") as f:
    json.dump(policy_v3, f, indent=2)


def ci(values):
    low, high = np.percentile(values, [2.5, 97.5])
    return {"mean": round(float(values.mean()), 4),
            "ci_95_low": round(float(low), 4), "ci_95_high": round(float(high), 4)}


metrics_v3 = {
    "test_period": "2015-07 to 2015-12",
    "test_loans": int(len(y["test"])),
    "actual_default_rate": round(float(y["test"].mean()), 4),
    "lightgbm": {
        "auc": round(float(comparison.loc["LightGBM v3", "auc"]), 4),
        "gini": round(float(comparison.loc["LightGBM v3", "gini"]), 4),
        "ks": round(float(comparison.loc["LightGBM v3", "ks"]), 4),
        "brier": round(float(brier_score_loss(y["test"], p_test)), 4),
        "avg_predicted": round(float(p_test.mean()), 4),
    },
    "logistic_regression_auc": round(float(comparison.loc["Logistic regression", "auc"]), 4),
    "lending_club_sub_grade_auc": round(float(comparison.loc["Lending Club sub_grade", "auc"]), 4),
    "lead_over_sub_grade": ci(lead_grade),
    "lightgbm_v2_auc": round(float(comparison.loc["LightGBM v2", "auc"]), 4),
    "lead_over_v2": ci(lead_v2),
}
with open("../results/metrics_v3.json", "w") as f:
    json.dump(metrics_v3, f, indent=2)

policy_table.to_csv("../results/policy_test_results_v3.csv")
print(json.dumps(metrics_v3, indent=2))

{
  "test_period": "2015-07 to 2015-12",
  "test_loans": 162236,
  "actual_default_rate": 0.1471,
  "lightgbm": {
    "auc": 0.7046,
    "gini": 0.4092,
    "ks": 0.2963,
    "brier": 0.1168,
    "avg_predicted": 0.1464
  },
  "logistic_regression_auc": 0.6871,
  "lending_club_sub_grade_auc": 0.6814,
  "lead_over_sub_grade": {
    "mean": 0.0231,
    "ci_95_low": 0.0205,
    "ci_95_high": 0.0262
  },
  "lightgbm_v2_auc": 0.7026,
  "lead_over_v2": {
    "mean": 0.002,
    "ci_95_low": 0.001,
    "ci_95_high": 0.0031
  }
}


In [6]:
typical = pd.read_parquet("../models/typical_applicant.parquet")
typical["emp_title"] = "manager"
typical.to_parquet("../models/typical_applicant_v3.parquet", index=False)

demo = pd.read_parquet("../models/demo_loans.parquet").merge(titles, on="id", how="left")
demo.to_parquet("../models/demo_loans_v3.parquet", index=False)

print("Demo loans:", len(demo), "| with a job title:", round(demo["emp_title"].notna().mean() * 100, 1), "%")

Demo loans: 2000 | with a job title: 92.4 %


In [7]:
one = model_v3.predict_proba(typical)[:, 1][0]
print(f"v3 risk for the typical applicant (manager): {one:.4f}")

v3_loaded = joblib.load("../models/lightgbm_v3_calibrated.joblib")
print("Saved model gives the same answer:",
      abs(v3_loaded.predict_proba(typical)[:, 1][0] - one) < 1e-12)

v3 risk for the typical applicant (manager): 0.0817
Saved model gives the same answer: True
